In [10]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("VoiceAgent.pdf")

documents = loader.load()


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text_splitter.split_documents(documents)
documents

[Document(metadata={'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36', 'creationdate': '2026-03-26T09:00:42+00:00', 'title': 'VoiceAgent_Assignment - Google Docs', 'moddate': '2026-03-26T09:00:42+00:00', 'source': 'VoiceAgent.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="ENGINEERING ASSIGNMENT \n Real-Time Multilingual Voice AI Agent \n Clinical Appointment Booking \n Preferred stack: Python · TypeScript \n Overview \n You are building a real-time voice AI agent for a digital healthcare platform. The agent's primary function \n is to book and manage clinical appointments through natural voice conversations — entirely without \n human intervention. \n The agent operates across three Indian languages (English, Hindi, Tamil), maintains awareness of patient \n context both within a session and across past interactions, and can proactively reach out to patients as \

In [12]:
#FAISS Vector Store
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# create embeddings (local)
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

db1 = FAISS.from_documents(documents,embeddings)

In [13]:
query = "Who are the stakeholders of the project?"
result = db1.similarity_search(query)
result[0].page_content

'Constraints \n –  Tool-calling and agent orchestration must be genuinely implemented — not simulated with \n hardcoded responses \n –  Reasoning traces must be visible and demonstrable \n We are evaluating systems thinking, latency awareness, and architectural clarity — not just whether the demo \n works. \n Confidential — Internal Use Only  |  Page  3'

In [14]:
from langchain_community.llms import Ollama

llm = Ollama(model="llama3.1:8b")
llm

C:\Users\pande\AppData\Local\Temp\ipykernel_20248\234902816.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1:8b")


Ollama(model='llama3.1:8b')

In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.
Think step by step before providing a detailed answer.
I will tip you $1000 if the user finds the answer helpful.
<context>
{context}
</context>
Question: {input}""")

In [16]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Replacement for create_stuff_documents_chain
document_chain = (
    {"context": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
retriever = db1.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F4AA56D960>, search_kwargs={})

In [ ]:
#Retriever-Document Chain
